# Event Sentiment Analysis

This notebook loads the event management CSV file, calculates a sentiment score from -100 to 100 for each event request, assigns a sentiment label, and explains the reason behind the score.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data").exists() and PROJECT_ROOT.name == "Script":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_PATH = DATA_DIR / "event_sentiment_analysis.csv"
DATA_DIR

PosixPath('/Users/hosnieh/Hosnieh_Folders/Projects/Sentiment_Analysis/Data')

In [3]:
def load_event_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    csv_files = sorted(data_dir.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    source_files = [path for path in csv_files if path.name != OUTPUT_PATH.name]
    if not source_files:
        raise FileNotFoundError(f"No source CSV files found in {data_dir}")

    if len(source_files) > 1:
        print("Multiple CSV files found. Loading the first one:")
        for csv_file in source_files:
            print(f"  - {csv_file.name}")

    csv_path = source_files[0]
    print(f"Loading data from: {csv_path}")
    return pd.read_csv(csv_path)


event_data = load_event_data()
event_data

Loading data from: /Users/hosnieh/Hosnieh_Folders/Projects/Sentiment_Analysis/Data/EventManagmentDataSet.csv


,SNO,Event,Faculty_id,Faculty_name,Faculty_Designation,Registers,Type,Request for Booking,Request for Cancelling,Requested Room No,Request Date,Requested Time,Duration,Room Available/Not Available,Accept/Reject
0,1,make a creative logo,FAC010102,John Doe,Associate Professor,100,Important,1,0,179,9/27/2022,3:00pm,120min,Available,Accept
1,2,Presentiong ideas on MSME,FAC010070,Jane Smith,Associate Professor,105,Very important,1,0,323,10/18/2022,3:00pm,120min,Available,Accept
2,3,Webinar on Web-Technologies,FAC010103,Alice Johnson,Associate Professor,77,Normal,1,0,279,10/25/2022,1:00pm,180min,Not Available,Reject
3,4,Group Discussion,FAC010077,Bob Brown,Associate Professor,60,Important,1,0,202,11/1/2022,2:00pm,90min,Available,Accept
4,5,Session on SIH,FAC010060,Carol Davis,Associate Professor,111,Very important,1,0,202,11/8/2022,3:00pm,90min,Available,Accept
5,6,Quiz-CODE KNACK WITH C,FAC010077,Bob Brown,Associate Professor,50,Very important,1,0,202,11/15/2022,2:00pm,180min,Available,Accept
6,7,JAM,FAC010103,Alice Johnson,Associate Professor,32,Normal,1,0,230,9/4/2023,3:30pm,90min,Available,Accept
7,8,QUIZ,FAC010077,Bob Brown,Associate Professor,28,Important,1,0,230,10/18/2022,3:30pm,90min,Available,Accept
8,9,WATCHING SITCOM,FAC010060,Carol Davis,Associate Professor,34,Normal,1,0,230,11/8/2022,3:30pm,90min,Available,Accept
9,10,Debate,FAC010070,Jane Smith,Associate Professor,15,Very important,1,0,209,2/2/2024,3:00pm,120min,Available,Accept


## Sentiment Scoring Method

The score is based on the request outcome, room availability, event importance, booking or cancellation request, registration count, and positive words in the event title. The final score is capped between -100 and 100.

In [4]:
def as_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def analyze_event_sentiment(row: pd.Series) -> pd.Series:
    score = 0
    reasons = []

    decision = as_text(row.get("Accept/Reject")).lower()
    availability = as_text(row.get("Room Available/Not Available")).lower()
    priority = as_text(row.get("Type")).lower()
    event_name = as_text(row.get("Event")).lower()
    booking_requested = int(row.get("Request for Booking", 0) or 0)
    cancellation_requested = int(row.get("Request for Cancelling", 0) or 0)
    registers = int(row.get("Registers", 0) or 0)

    if decision == "accept":
        score += 45
        reasons.append("accepted request")
    elif decision == "reject":
        score -= 55
        reasons.append("request was rejected")

    if availability == "available":
        score += 25
        reasons.append("room is available")
    elif availability == "not available":
        score -= 35
        reasons.append("room is not available")

    if priority == "very important":
        score += 20
        reasons.append("event is marked very important")
    elif priority == "important":
        score += 12
        reasons.append("event is marked important")
    elif priority == "normal":
        reasons.append("event priority is normal")

    if booking_requested:
        score += 5
        reasons.append("booking was requested")

    if cancellation_requested:
        score -= 25
        reasons.append("cancellation was requested")

    if registers >= 100:
        score += 15
        reasons.append("very high registration count")
    elif registers >= 50:
        score += 8
        reasons.append("strong registration count")
    elif registers < 20:
        score -= 5
        reasons.append("low registration count")

    positive_event_words = {
        "creative",
        "ideas",
        "webinar",
        "discussion",
        "session",
        "quiz",
        "debate",
        "review",
        "contest",
        "greeting",
        "singing",
    }
    matched_words = sorted(word for word in positive_event_words if word in event_name)
    if matched_words:
        score += min(10, len(matched_words) * 5)
        reasons.append(f"positive event keywords: {', '.join(matched_words)}")

    score = max(-100, min(100, score))

    if score > 20:
        label = "Positive"
    elif score < -20:
        label = "Negative"
    else:
        label = "Neutral"

    if label == "Negative":
        explanation = "Negative because " + "; ".join(reasons)
    elif label == "Positive":
        explanation = "Positive because " + "; ".join(reasons)
    else:
        explanation = "Neutral because " + "; ".join(reasons)

    return pd.Series(
        {
            "Sentiment Score": score,
            "Sentiment Label": label,
            "Sentiment Reason": explanation,
        }
    )

In [5]:
sentiment_columns = event_data.apply(analyze_event_sentiment, axis=1)
sentiment_data = pd.concat([event_data, sentiment_columns], axis=1)

sentiment_data[
    [
        "Event",
        "Type",
        "Registers",
        "Room Available/Not Available",
        "Accept/Reject",
        "Sentiment Score",
        "Sentiment Label",
        "Sentiment Reason",
    ]
]

,Event,Type,Registers,Room Available/Not Available,Accept/Reject,Sentiment Score,Sentiment Label,Sentiment Reason
0,make a creative logo,Important,100,Available,Accept,100,Positive,Positive because accepted request; room is ava...
1,Presentiong ideas on MSME,Very important,105,Available,Accept,100,Positive,Positive because accepted request; room is ava...
2,Webinar on Web-Technologies,Normal,77,Not Available,Reject,-72,Negative,Negative because request was rejected; room is...
3,Group Discussion,Important,60,Available,Accept,100,Positive,Positive because accepted request; room is ava...
4,Session on SIH,Very important,111,Available,Accept,100,Positive,Positive because accepted request; room is ava...
5,Quiz-CODE KNACK WITH C,Very important,50,Available,Accept,100,Positive,Positive because accepted request; room is ava...
6,JAM,Normal,32,Available,Accept,75,Positive,Positive because accepted request; room is ava...
7,QUIZ,Important,28,Available,Accept,92,Positive,Positive because accepted request; room is ava...
8,WATCHING SITCOM,Normal,34,Available,Accept,75,Positive,Positive because accepted request; room is ava...
9,Debate,Very important,15,Available,Accept,95,Positive,Positive because accepted request; room is ava...


In [6]:
sentiment_data["Sentiment Label"].value_counts()

Sentiment Label
Positive    13
Negative     2
Name: count, dtype: int64

In [8]:
sentiment_data.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

PosixPath('/Users/hosnieh/Hosnieh_Folders/Projects/Sentiment_Analysis/Data/event_sentiment_analysis.csv')